In [0]:
SELECT
  pr.series_id,
  'PR' as survery_code,
  pr.season_code,
  pr.sector_code,
  pr.class_code,
  pr.measure_code,
  pr.duration_code,
  'Major Sector Productivity and Costs Index' as survey_abbreviation,
  s.seasonal_text,
  sec.sector_name,
  c.class_text,
  m.measure_text,
  d.duration_text,
  pr.year,
  pr.total
FROM
  (
    SELECT
      *,
      SUBSTRING(series_id, 3, 1) AS season_code,
      SUBSTRING(series_id, 4, 4) AS sector_code,
      SUBSTRING(series_id, 8, 1) AS class_code,
      SUBSTRING(series_id, 9, 2) AS measure_code,
      SUBSTRING(series_id, 11, 1) AS duration_code,
      row_number() OVER (PARTITION BY series_id ORDER BY total DESC) AS _rn
    FROM
      (
        SELECT
          series_id,
          year,
          SUM(value) AS total
        FROM
          rearc.silver.all_data
        GROUP BY
          series_id,
          year
      )
  ) pr
LEFT JOIN rearc.silver.seasonal  s   ON pr.season_code    = s.seasonal_code
LEFT JOIN rearc.silver.sector    sec ON pr.sector_code    = sec.sector_code
LEFT JOIN rearc.silver.class     c   ON pr.class_code     = c.class_code
LEFT JOIN rearc.silver.measure   m   ON pr.measure_code   = m.measure_code
LEFT JOIN rearc.silver.duration  d   ON pr.duration_code  = d.duration_code
WHERE
  pr._rn = 1
ORDER BY
  pr.series_id,
  pr.total DESC

In [0]:
SELECT d.series_id, d.year, d.period, p.population, d.value FROM rearc.silver.all_data d
LEFT JOIN rearc.silver.population p ON d.year = p.year
WHERE d.series_id = 'PRS30006032' AND d.period = 'Q01'
ORDER BY year